# 7.3 隐私与安全

端侧AI部署中的隐私保护和安全防护是产业落地的硬性要求。医疗、金融、政务等场景对数据隐私有严格法规约束，模型安全也面临对抗攻击、模型窃取等威胁。

## 为什么端侧AI需要隐私与安全？

1. **法规合规**：GDPR、CCPA、个人信息保护法等要求用户数据本地处理
2. **数据敏感性**：医疗诊断、金融风控、人脸识别等数据高度敏感
3. **模型资产保护**：训练成本高昂的模型面临窃取和逆向工程风险
4. **对抗攻击**：恶意输入可导致模型输出错误结果
5. **供应链安全**：模型文件可能被植入后门

## 端侧AI安全威胁模型

| 威胁 | 攻击方式 | 影响 | 防御 |
|------|---------|------|------|
| **数据泄露** | 中间人窃听端云通信 | 用户隐私暴露 | TLS加密+本地处理 |
| **模型窃取** | API查询逆向模型参数 | IP损失 | 水印+差分隐私 |
| **对抗攻击** | 添加不可见扰动到输入 | 错误输出 | 对抗训练+输入检测 |
| **后门攻击** | 训练时植入触发器 | 特定输入触发恶意行为 | 模型审计+校验 |
| **模型篡改** | 替换端侧模型文件 | 功能异常 | 签名校验+TEE |
| **成员推断** | 判断数据是否在训练集中 | 隐私泄露 | 差分隐私训练 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import hashlib
import json
import time
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple
from collections import OrderedDict

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

### 联邦学习（Federated Learning）

#### 什么是联邦学习？

联邦学习允许多个设备在不共享原始数据的前提下协同训练模型。每个设备在本地训练，仅上传模型梯度（或参数更新）到服务器聚合。

#### 为什么用联邦学习？

1. **数据不出域**：原始数据保留在端侧，仅上传梯度
2. **法规合规**：满足GDPR等隐私法规的数据本地化要求
3. **个性化**：每个设备可以保留本地个性化模型

#### 联邦学习的数学原理

**FedAvg算法**：
$$\theta_{t+1} = \sum_{k=1}^{K} \frac{n_k}{n} \theta_{t+1}^{k}$$
其中$\theta_{t+1}^{k}$是第$k$个客户端在本地数据上训练$E$个epoch后的模型参数，$n_k$是第$k$个客户端的数据量。

**安全性分析**：
- 原始梯度可能泄露训练数据信息（梯度反演攻击）
- 防御：梯度加噪（差分隐私）、梯度压缩（减少信息量）、安全聚合（加密梯度）

#### 产业级联邦学习框架

| 框架 | 开发者 | 特点 |
|------|--------|------|
| **PySyft** | OpenMined | 隐私计算+联邦学习 |
| **FATE** | 微众银行 | 金融级联邦学习 |
| **Flower** | Adap | 轻量级联邦学习 |
| **TensorFlow Federated** | Google | TFF框架 |

In [ ]:
@dataclass
class ClientUpdate:
    client_id: int
    num_samples: int
    gradient: Dict[str, torch.Tensor]
    noise_added: bool = False
    noise_scale: float = 0.0

class FederatedServer:
    """联邦学习服务器: FedAvg聚合"""
    def __init__(self, model: nn.Module):
        self.global_model = model
        self.round = 0
        self.history = {'round': [], 'avg_loss': [], 'num_clients': []}

    def aggregate(self, updates: List[ClientUpdate]) -> Dict[str, torch.Tensor]:
        """FedAvg加权聚合"""
        total_samples = sum(u.num_samples for u in updates)
        aggregated = OrderedDict()
        for key in updates[0].gradient.keys():
            weighted_sum = sum(
                u.num_samples / total_samples * u.gradient[key]
                for u in updates
            )
            aggregated[key] = weighted_sum
        return aggregated

    def apply_update(self, aggregated: Dict[str, torch.Tensor], lr=0.01):
        """应用聚合后的梯度更新全局模型"""
        state_dict = self.global_model.state_dict()
        for key, grad in aggregated.items():
            if key in state_dict:
                state_dict[key] = state_dict[key] - lr * grad
        self.global_model.load_state_dict(state_dict)
        self.round += 1

class FederatedClient:
    """联邦学习客户端: 本地训练+梯度加噪"""
    def __init__(self, client_id: int, model: nn.Module, data_size: int,
                 dp_enabled: bool = False, dp_epsilon: float = 1.0, dp_delta: float = 1e-5):
        self.client_id = client_id
        self.model = model
        self.data_size = data_size
        self.dp_enabled = dp_enabled
        self.dp_epsilon = dp_epsilon
        self.dp_delta = dp_delta

    def local_train(self, x: torch.Tensor, y: torch.Tensor, epochs: int = 3, lr: float = 0.01) -> ClientUpdate:
        """本地训练并返回梯度更新"""
        original_state = {k: v.clone() for k, v in self.model.state_dict().items()}
        optimizer = torch.optim.SGD(self.model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        for _ in range(epochs):
            optimizer.zero_grad()
            output = self.model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
        gradient = OrderedDict()
        noise_scale = 0.0
        for key, new_val in self.model.state_dict().items():
            grad = original_state[key] - new_val
            if self.dp_enabled and grad.is_floating_point():
                grad_norm = grad.norm().item()
                clip_bound = 1.0
                if grad_norm > clip_bound:
                    grad = grad * (clip_bound / grad_norm)
                sensitivity = clip_bound / self.data_size
                sigma = sensitivity * np.sqrt(2 * np.log(1.25 / self.dp_delta)) / self.dp_epsilon
                noise = torch.randn_like(grad) * sigma
                grad = grad + noise
                noise_scale = sigma
            gradient[key] = grad
        self.model.load_state_dict(original_state)
        return ClientUpdate(
            client_id=self.client_id, num_samples=self.data_size,
            gradient=gradient, noise_added=self.dp_enabled, noise_scale=noise_scale
        )

class SimpleClassifier(nn.Module):
    def __init__(self, dim=32, n_classes=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, 16), nn.ReLU(), nn.Linear(16, n_classes))
    def forward(self, x):
        return self.net(x)

global_model = SimpleClassifier(dim=32, n_classes=5)
server = FederatedServer(global_model)

n_clients = 5
clients = []
for i in range(n_clients):
    local_model = SimpleClassifier(dim=32, n_classes=5)
    local_model.load_state_dict(global_model.state_dict())
    client = FederatedClient(i, local_model, data_size=100+i*20, dp_enabled=(i % 2 == 0))
    clients.append(client)

print("=== 联邦学习模拟 (5个客户端, 3轮) ===")
for round_idx in range(3):
    updates = []
    for client in clients:
        x = torch.randn(client.data_size, 32)
        y = torch.randint(0, 5, (client.data_size,))
        update = client.local_train(x, y, epochs=2)
        updates.append(update)
    aggregated = server.aggregate(updates)
    server.apply_update(aggregated, lr=0.01)
    dp_clients = sum(1 for u in updates if u.noise_added)
    print(f"  Round {round_idx+1}: {len(updates)}个客户端, {dp_clients}个启用DP, "
          f"噪声σ={updates[0].noise_scale:.4f}")

print(f"\n--- 联邦学习安全性分析 ---")
print(f"1. 梯度反演风险: 原始梯度可还原训练数据(见Zhu et al. DLG)")
print(f"2. DP防御: 梯度裁剪+高斯噪声, ε越小隐私保护越强但精度越低")
print(f"3. 安全聚合: 同态加密梯度, 服务器无法看到单个客户端梯度")
print(f"4. 产业实践: 医疗/金融场景必须启用DP(ε≤10)或安全聚合")

---
### 差分隐私（Differential Privacy）

#### 什么是差分隐私？

差分隐私提供数学上可证明的隐私保证：无论攻击者拥有多少背景知识，都无法确定某个个体是否在数据集中。

#### 差分隐私的数学定义

$(\epsilon, \delta)$-差分隐私：对于任意两个相邻数据集$D$和$D'$（仅差一条记录），以及任意输出集合$S$：
$$P[\mathcal{M}(D) \in S] \leq e^{\epsilon} \cdot P[\mathcal{M}(D') \in S] + \delta$$

- $\epsilon$：隐私预算，越小隐私保护越强（典型值0.1-10）
- $\delta$：失败概率，通常设为$1/n^2$（$n$为数据集大小）

#### DP-SGD算法

1. **梯度裁剪**：$\tilde{g}_i = g_i \cdot \min(1, C/\|g_i\|_2)$，限制每个样本的梯度贡献
2. **聚合加噪**：$\bar{g} = \frac{1}{B}\left(\sum_i \tilde{g}_i + \mathcal{N}(0, \sigma^2 C^2 I)\right)$
3. **隐私预算计算**：通过Rényi DP或Moments Accountant跟踪累积$\epsilon$

#### 隐私预算与精度权衡

| ε | 隐私保护 | 精度影响 | 适用场景 |
|---|---------|---------|----------|
| <1 | 极强 | 严重下降 | 理论研究 |
| 1-3 | 强 | 中等下降 | 医疗/金融 |
| 3-8 | 中等 | 轻微下降 | 通用场景 |
| >8 | 弱 | 几乎无损 | 非敏感场景 |

In [ ]:
class DPSGDOptimizer:
    """差分隐私SGD优化器: 梯度裁剪+高斯噪声"""
    def __init__(self, model, lr=0.01, max_grad_norm=1.0, noise_multiplier=1.0,
                 dataset_size=1000, batch_size=32, target_delta=1e-5):
        self.model = model
        self.lr = lr
        self.max_grad_norm = max_grad_norm
        self.noise_multiplier = noise_multiplier
        self.dataset_size = dataset_size
        self.batch_size = batch_size
        self.target_delta = target_delta
        self.steps = 0
        self.epsilon_spent = 0.0

    def _clip_gradients(self):
        """逐样本梯度裁剪"""
        total_norm = 0.0
        for p in self.model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        clip_coef = min(1.0, self.max_grad_norm / (total_norm + 1e-8))
        for p in self.model.parameters():
            if p.grad is not None:
                p.grad.data.mul_(clip_coef)

    def _add_noise(self):
        """添加高斯噪声"""
        for p in self.model.parameters():
            if p.grad is not None:
                noise = torch.randn_like(p.grad) * self.max_grad_norm * self.noise_multiplier / self.batch_size
                p.grad.data.add_(noise)

    def _compute_epsilon(self, steps):
        """简化隐私预算计算 (Moments Accountant近似)"""
        q = self.batch_size / self.dataset_size
        sigma = self.noise_multiplier
        if sigma <= 0:
            return float('inf')
        c2 = 2 * np.log(1.25 / self.target_delta)
        eps_per_step = q * np.sqrt(c2) / sigma
        return eps_per_step * np.sqrt(steps)

    def step(self):
        self._clip_gradients()
        self._add_noise()
        for p in self.model.parameters():
            if p.grad is not None:
                p.data -= self.lr * p.grad.data
        self.steps += 1
        self.epsilon_spent = self._compute_epsilon(self.steps)

    def zero_grad(self):
        self.model.zero_grad()

model = SimpleClassifier(dim=32, n_classes=5)
dp_optimizer = DPSGDOptimizer(model, lr=0.01, max_grad_norm=1.0, noise_multiplier=1.0,
                               dataset_size=1000, batch_size=32)

print("=== 差分隐私训练模拟 ===")
print(f"\n--- DP-SGD训练过程 ---")
print(f"{'Step':<8} {'Loss':<10} {'ε(累计)':<12} {'噪声强度':<12} {'梯度范数'}")
print("-" * 55)

criterion = nn.CrossEntropyLoss()
for step in range(20):
    x = torch.randn(32, 32)
    y = torch.randint(0, 5, (32,))
    dp_optimizer.zero_grad()
    output = model(x)
    loss = criterion(output, y)
    loss.backward()
    grad_norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
    dp_optimizer.step()
    if step % 5 == 0:
        print(f"{step:<8} {loss.item():<10.4f} {dp_optimizer.epsilon_spent:<12.4f} "
              f"{dp_optimizer.noise_multiplier:<12.1f} {grad_norm:.4f}")

print(f"\n--- 不同噪声强度的隐私-精度权衡 ---")
print(f"{'σ(噪声)':<10} {'ε(100步)':<12} {'隐私等级':<12} {'预期精度损失'}")
print("-" * 50)
for sigma in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    temp_opt = DPSGDOptimizer(SimpleClassifier(32, 5), noise_multiplier=sigma, dataset_size=1000, batch_size=32)
    eps = temp_opt._compute_epsilon(100)
    level = '极强' if eps < 1 else '强' if eps < 3 else '中等' if eps < 8 else '弱'
    acc_loss = '<5%' if sigma < 0.5 else '5-15%' if sigma < 2 else '15-30%' if sigma < 5 else '>30%'
    print(f"{sigma:<10} {eps:<12.2f} {level:<12} {acc_loss}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 医疗场景: ε≤3, σ≥2.0, 需要隐私预算审计")
print(f"2. 金融场景: ε≤8, σ≥1.0, 需要合规认证")
print(f"3. 通用场景: ε≤10, σ≥0.5, 可选启用")
print(f"4. 隐私预算是消耗品: 训练步数越多ε越大，需合理分配")
print(f"5. DP与联邦学习结合: Local DP(客户端加噪) + 安全聚合")

---
### 模型水印（Model Watermarking）

#### 什么是模型水印？

在模型参数中嵌入可验证的标记，用于证明模型所有权和检测模型窃取。

#### 为什么需要模型水印？

1. **IP保护**：训练成本高昂的模型可能被窃取或复制
2. **溯源**：验证部署的模型是否为授权版本
3. **合规**：法规要求AI模型可溯源

#### 水印方法

| 方法 | 原理 | 鲁棒性 | 对精度影响 |
|------|------|--------|----------|
| **后门水印** | 特定触发样本→特定输出 | 高 | 低 |
| **参数水印** | 在权重中编码信息 | 中 | 极低 |
| **输出分布水印** | 修改特定输入的输出分布 | 中 | 低 |

#### 后门水印的数学原理

训练时注入触发模式$(x_t, y_t)$：
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{task}} + \lambda \cdot \mathcal{L}_{\text{watermark}}$$
其中$\mathcal{L}_{\text{watermark}} = \text{CE}(f(x_t), y_t)$，$x_t$是触发输入，$y_t$是目标标签。

验证时：输入$x_t$，检查输出是否为$y_t$。匹配则证明模型所有权。

In [ ]:
class ModelWatermark:
    """模型水印: 后门水印+参数水印"""
    def __init__(self, trigger_pattern: torch.Tensor, target_label: int, n_trigger_samples: int = 10):
        self.trigger_pattern = trigger_pattern
        self.target_label = target_label
        self.n_trigger_samples = n_trigger_samples
        self.trigger_set_x = None
        self.trigger_set_y = None
        self._generate_trigger_set()

    def _generate_trigger_set(self):
        """生成触发样本集"""
        base = torch.randn(self.n_trigger_samples, *self.trigger_pattern.shape)
        self.trigger_set_x = base + self.trigger_pattern
        self.trigger_set_y = torch.full((self.n_trigger_samples,), self.target_label, dtype=torch.long)

    def get_trigger_loss(self, model: nn.Module) -> torch.Tensor:
        """计算水印损失"""
        output = model(self.trigger_set_x)
        return F.cross_entropy(output, self.trigger_set_y)

    def verify(self, model: nn.Module, threshold: float = 0.9) -> Dict:
        """验证水印是否存在"""
        with torch.no_grad():
            output = model(self.trigger_set_x)
            preds = output.argmax(dim=-1)
            match_rate = (preds == self.target_label).float().mean().item()
        return {
            'watermark_detected': match_rate >= threshold,
            'match_rate': match_rate,
            'threshold': threshold,
            'confidence': 'high' if match_rate > 0.95 else 'medium' if match_rate > 0.8 else 'low',
        }

class ParameterWatermark:
    """参数水印: 在模型权重中嵌入签名"""
    def __init__(self, signature: str = 'EDGE_AI_SECURE'):
        self.signature = signature
        self.signature_hash = hashlib.sha256(signature.encode()).hexdigest()[:16]

    def embed(self, model: nn.Module, positions: Optional[List[str]] = None) -> Dict:
        """在指定参数位置嵌入签名"""
        state_dict = model.state_dict()
        if positions is None:
            positions = [k for k in state_dict.keys() if 'weight' in k][:3]
        embedded_info = {}
        for i, key in enumerate(positions):
            param = state_dict[key]
            hash_bits = bin(int(self.signature_hash, 16))[2:]
            bit_pos = i % len(hash_bits)
            bit_val = int(hash_bits[bit_pos])
            mask = torch.ones_like(param) * 1e-5
            mask.flat[bit_pos] = bit_val * 1e-4
            state_dict[key] = param + mask
            embedded_info[key] = {'bit_pos': bit_pos, 'bit_val': bit_val}
        model.load_state_dict(state_dict)
        return embedded_info

    def verify(self, model: nn.Module, positions: List[str], embedded_info: Dict) -> bool:
        """验证参数签名"""
        state_dict = model.state_dict()
        for key in positions:
            if key not in embedded_info:
                return False
        return True

model = SimpleClassifier(dim=32, n_classes=5)

trigger = torch.randn(32) * 0.1
watermark = ModelWatermark(trigger, target_label=4, n_trigger_samples=20)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

print("=== 模型水印训练与验证 ===")
print(f"\n--- 水印训练 (5轮) ---")
for epoch in range(5):
    x = torch.randn(32, 32)
    y = torch.randint(0, 5, (32,))
    optimizer.zero_grad()
    task_loss = criterion(model(x), y)
    wm_loss = watermark.get_trigger_loss(model)
    total_loss = task_loss + 0.5 * wm_loss
    total_loss.backward()
    optimizer.step()
    result = watermark.verify(model)
    print(f"  Epoch {epoch+1}: 任务损失={task_loss.item():.4f}, 水印损失={wm_loss.item():.4f}, "
          f"匹配率={result['match_rate']:.1%}")

print(f"\n--- 水印验证结果 ---")
result = watermark.verify(model)
print(f"  水印检测: {'✓ 存在' if result['watermark_detected'] else '✗ 不存在'}")
print(f"  匹配率: {result['match_rate']:.1%}")
print(f"  置信度: {result['confidence']}")

print(f"\n--- 参数水印 ---")
param_wm = ParameterWatermark('EDGE_AI_SECURE_2024')
embedded = param_wm.embed(model)
print(f"  签名: {param_wm.signature}")
print(f"  哈希: {param_wm.signature_hash}")
print(f"  嵌入位置: {list(embedded.keys())}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 后门水印: 触发样本需保密，公开则可被绕过")
print(f"2. 参数水印: 鲁棒性较低，微调可能破坏，适合快速验证")
print(f"3. 水印组合: 后门水印(强验证)+参数水印(快速检测)组合使用")
print(f"4. 法律效力: 水印可作为IP侵权证据，但需独立第三方验证")
print(f"5. 模型注册: 训练完成后在模型注册中心登记水印信息")

---
### 可信执行环境（TEE）

#### 什么是TEE？

可信执行环境（Trusted Execution Environment）是处理器中的安全区域，保证内部代码和数据的机密性和完整性，即使操作系统被攻破也无法访问TEE内部。

#### 主流TEE技术

| TEE | 厂商 | 适用平台 | 安全等级 |
|-----|------|---------|----------|
| **Intel SGX** | Intel | 服务器/PC | 高 (Enclave隔离) |
| **ARM TrustZone** | ARM | 移动端/嵌入式 | 中 (安全世界隔离) |
| **Apple Secure Enclave** | Apple | iPhone/Mac | 高 (独立协处理器) |
| **AMD SEV** | AMD | 服务器 | 中 (内存加密) |
| **AWS Nitro Enclaves** | AWS | 云端 | 高 (虚拟Enclave) |

#### TEE在端侧AI中的应用

1. **模型保护**：模型权重存储在TEE中，防止窃取
2. **推理保护**：推理过程在TEE中执行，输入输出加密
3. **密钥管理**：模型解密密钥存储在TEE中
4. **远程认证**：证明端侧运行的是未篡改的模型

#### TEE的性能开销

- **上下文切换**：普通世界↔安全世界切换开销约$10\mu s$
- **内存限制**：SGX Enclave内存通常128MB，大模型需分页
- **加密开销**：内存加密/解密约5-10%性能损失
- **ECALL/OCALL**：跨世界函数调用开销约$1-10\mu s$

In [ ]:
@dataclass
class AttestationResult:
    is_valid: bool
    tee_type: str
    model_hash: str
    timestamp: float
    details: str

class TEESimulator:
    """TEE模拟器: 模拟可信执行环境的安全推理"""
    def __init__(self, tee_type='ARM_TrustZone', secure_memory_mb=128):
        self.tee_type = tee_type
        self.secure_memory_mb = secure_memory_mb
        self.encrypted_models: Dict[str, bytes] = {}
        self.model_hashes: Dict[str, str] = {}
        self.attestation_log: List[AttestationResult] = []
        self.context_switch_overhead_us = 10
        self.encryption_overhead_pct = 0.08

    def _hash_model(self, state_dict: Dict[str, torch.Tensor]) -> str:
        """计算模型哈希"""
        hasher = hashlib.sha256()
        for key in sorted(state_dict.keys()):
            hasher.update(state_dict[key].numpy().tobytes())
        return hasher.hexdigest()[:16]

    def _xor_encrypt(self, data: bytes, key: int) -> bytes:
        """简单XOR加密(演示用, 产业级应使用AES-256)"""
        return bytes(b ^ (key & 0xFF) for b in data)

    def load_model(self, model_id: str, model: nn.Module):
        """加密并加载模型到TEE"""
        state_dict = model.state_dict()
        model_hash = self._hash_model(state_dict)
        buffer = BytesIO()
        torch.save(state_dict, buffer)
        buffer.seek(0)
        model_bytes = buffer.getvalue()
        encrypted = self._xor_encrypt(model_bytes, 0xAB)
        self.encrypted_models[model_id] = encrypted
        self.model_hashes[model_id] = model_hash
        model_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
        return {'model_id': model_id, 'hash': model_hash, 'size_mb': model_size_mb,
                'fits_tee': model_size_mb <= self.secure_memory_mb}

    def _xor_decrypt(self, data: bytes, key: int) -> bytes:
        """XOR解密(与加密同一操作, 产业级应使用AES-256-GCM)"""
        return bytes(b ^ (key & 0xFF) for b in data)

    def _decrypt_model(self, model_id: str) -> Dict[str, torch.Tensor]:
        """从加密存储解密并反序列化模型"""
        encrypted = self.encrypted_models[model_id]
        decrypted = self._xor_decrypt(encrypted, 0xAB)
        buffer = BytesIO(decrypted)
        buffer.seek(0)
        state_dict = torch.load(buffer, weights_only=True)
        return state_dict

    def secure_inference(self, model_id: str, model: nn.Module, x: torch.Tensor) -> Dict:
        """在TEE中执行安全推理: 解密模型→完整性校验→推理"""
        if model_id not in self.encrypted_models:
            return {'error': 'Model not loaded in TEE'}
        decrypt_start = time.perf_counter()
        decrypted_state_dict = self._decrypt_model(model_id)
        decrypt_time = (time.perf_counter() - decrypt_start) * 1000
        restored_hash = self._hash_model(decrypted_state_dict)
        integrity_ok = restored_hash == self.model_hashes[model_id]
        if not integrity_ok:
            return {'error': 'Model integrity check failed', 'expected': self.model_hashes[model_id],
                    'actual': restored_hash}
        model.load_state_dict(decrypted_state_dict)
        start = time.perf_counter()
        with torch.no_grad():
            output = model(x)
        inference_time = (time.perf_counter() - start) * 1000
        tee_overhead = inference_time * self.encryption_overhead_pct
        total_time = inference_time + tee_overhead + self.context_switch_overhead_us / 1000 + decrypt_time
        return {
            'model_id': model_id, 'integrity_ok': integrity_ok,
            'decrypt_ms': decrypt_time, 'inference_ms': inference_time,
            'tee_overhead_ms': tee_overhead, 'total_ms': total_time,
            'output_shape': list(output.shape),
        }

    def remote_attestation(self, model_id: str) -> AttestationResult:
        """远程认证: 证明TEE中运行的是正确模型"""
        result = AttestationResult(
            is_valid=model_id in self.encrypted_models,
            tee_type=self.tee_type,
            model_hash=self.model_hashes.get(model_id, 'unknown'),
            timestamp=time.time(),
            details=f'Attestation passed for {model_id}' if model_id in self.encrypted_models else 'Model not found'
        )
        self.attestation_log.append(result)
        return result

from io import BytesIO

tee = TEESimulator(tee_type='ARM_TrustZone', secure_memory_mb=128)
model = SimpleClassifier(dim=32, n_classes=5)

load_result = tee.load_model('classifier_v1', model)
print(f"=== TEE安全推理模拟 ===")
print(f"TEE类型: {tee.tee_type}")
print(f"安全内存: {tee.secure_memory_mb}MB")
print(f"模型加载: {load_result['model_id']}, 哈希={load_result['hash']}, "
      f"大小={load_result['size_mb']:.2f}MB, 适配TEE={'是' if load_result['fits_tee'] else '否'}")

x = torch.randn(1, 32)
result = tee.secure_inference('classifier_v1', model, x)
print(f"\n--- 安全推理结果 ---")
print(f"  完整性校验: {'✓ 通过' if result['integrity_ok'] else '✗ 失败'}")
print(f"  模型解密: {result['decrypt_ms']:.2f}ms")
print(f"  推理延迟: {result['inference_ms']:.2f}ms")
print(f"  TEE开销: {result['tee_overhead_ms']:.2f}ms ({tee.encryption_overhead_pct*100:.0f}%)")
print(f"  上下文切换: {tee.context_switch_overhead_us}μs")
print(f"  总延迟: {result['total_ms']:.2f}ms")

attestation = tee.remote_attestation('classifier_v1')
print(f"\n--- 远程认证 ---")
print(f"  认证结果: {'✓ 通过' if attestation.is_valid else '✗ 失败'}")
print(f"  TEE类型: {attestation.tee_type}")
print(f"  模型哈希: {attestation.model_hash}")

tampered = tee.secure_inference('unknown_model', model, x)
print(f"\n--- 篡改检测 ---")
print(f"  未注册模型推理: {tampered.get('error', '成功')}")

print(f"\n=== 产业实践要点 ===")
print(f"1. ARM TrustZone: 移动端标配, 适合小模型(<100MB)")
print(f"2. Intel SGX: 服务器端, 128MB Enclave需分页加载大模型")
print(f"3. 远程认证: 云端验证端侧TEE状态, 确保模型未被篡改")
print(f"4. 性能开销: TEE推理比普通推理慢5-15%, 安全换性能")
print(f"5. 密钥管理: 模型解密密钥存储在TEE中, OS无法读取")

---
### 安全推理协议

#### 什么是安全推理？

安全推理确保端云协同推理过程中，用户的原始输入和中间特征不被窃取或篡改。

#### 安全推理的关键环节

| 环节 | 威胁 | 防御 |
|------|------|------|
| **输入传输** | 中间人窃听 | TLS 1.3加密 |
| **中间特征** | 梯度反演还原输入 | 特征扰动+噪声注入 |
| **模型文件** | 窃取/篡改 | 加密存储+签名校验 |
| **推理结果** | 结果篡改 | 数字签名+MAC |
| **日志数据** | 敏感信息泄露 | 数据脱敏+匿名化 |

#### 端云安全通信协议

```
端侧                              云端
  │                                │
  │──── TLS 1.3握手 ────────────→│
  │←─── 证书验证 ────────────────│
  │                                │
  │──── 加密输入/中间特征 ──────→│
  │←─── 加密推理结果 ───────────│
  │                                │
  │──── 结果签名验证 ───────────→│
```

In [ ]:
class SecureInferenceProtocol:
    """安全推理协议: 模拟端云安全通信"""
    def __init__(self):
        self.session_key = np.random.bytes(32)
        self.nonce_counter = 0
        self.protocol_log = []

    def _compute_mac(self, data: bytes) -> str:
        """计算消息认证码(MAC)"""
        h = hashlib.sha256()
        h.update(self.session_key)
        h.update(data)
        h.update(str(self.nonce_counter).encode())
        return h.hexdigest()[:16]

    def encrypt_feature(self, feature: torch.Tensor, noise_sigma: float = 0.0) -> Dict:
        """加密中间特征: 量化+可选噪声+MAC"""
        self.nonce_counter += 1
        if noise_sigma > 0:
            feature = feature + torch.randn_like(feature) * noise_sigma
        compressed = feature.half()
        data_bytes = compressed.numpy().tobytes()
        mac = self._compute_mac(data_bytes)
        return {'data': compressed, 'mac': mac, 'nonce': self.nonce_counter,
                'noise_sigma': noise_sigma, 'size_bytes': len(data_bytes)}

    def decrypt_and_verify(self, encrypted: Dict) -> Tuple[torch.Tensor, bool]:
        """解密并验证MAC"""
        data_bytes = encrypted['data'].numpy().tobytes()
        expected_mac = self._compute_mac(data_bytes)
        integrity_ok = expected_mac == encrypted['mac']
        feature = encrypted['data'].float()
        return feature, integrity_ok

    def sign_result(self, result: torch.Tensor) -> Dict:
        """对推理结果签名"""
        self.nonce_counter += 1
        result_bytes = result.numpy().tobytes()
        mac = self._compute_mac(result_bytes)
        return {'result': result, 'mac': mac, 'nonce': self.nonce_counter}

    def verify_result(self, signed: Dict) -> Tuple[torch.Tensor, bool]:
        """验证推理结果签名"""
        result_bytes = signed['result'].numpy().tobytes()
        expected_mac = self._compute_mac(result_bytes)
        integrity_ok = expected_mac == signed['mac']
        return signed['result'], integrity_ok

class ModelIntegrityChecker:
    """模型完整性校验器"""
    def __init__(self):
        self.registered_hashes: Dict[str, str] = {}

    def register_model(self, model_id: str, model: nn.Module):
        """注册模型哈希"""
        state_dict = model.state_dict()
        hasher = hashlib.sha256()
        for key in sorted(state_dict.keys()):
            hasher.update(state_dict[key].numpy().tobytes())
        self.registered_hashes[model_id] = hasher.hexdigest()[:16]

    def verify_model(self, model_id: str, model: nn.Module) -> Dict:
        """验证模型完整性"""
        state_dict = model.state_dict()
        hasher = hashlib.sha256()
        for key in sorted(state_dict.keys()):
            hasher.update(state_dict[key].numpy().tobytes())
        current_hash = hasher.hexdigest()[:16]
        expected_hash = self.registered_hashes.get(model_id, 'unknown')
        return {
            'model_id': model_id, 'integrity_ok': current_hash == expected_hash,
            'expected_hash': expected_hash, 'current_hash': current_hash,
        }

protocol = SecureInferenceProtocol()
checker = ModelIntegrityChecker()
model = SimpleClassifier(dim=32, n_classes=5)
checker.register_model('classifier_v1', model)

print("=== 安全推理协议模拟 ===")
feature = torch.randn(1, 16, 32)

print(f"\n--- 端侧加密中间特征 ---")
for noise_sigma in [0.0, 0.01, 0.05, 0.1]:
    encrypted = protocol.encrypt_feature(feature, noise_sigma=noise_sigma)
    decrypted, ok = protocol.decrypt_and_verify(encrypted)
    diff = (feature - decrypted).abs().max().item()
    print(f"  σ={noise_sigma}: MAC={'✓' if ok else '✗'}, 传输={encrypted['size_bytes']}bytes, "
          f"最大差异={diff:.4f}")

print(f"\n--- 推理结果签名验证 ---")
result = torch.randn(1, 5)
signed = protocol.sign_result(result)
verified_result, verified = protocol.verify_result(signed)
print(f"  签名验证: {'✓ 通过' if verified else '✗ 失败'}")
print(f"  结果一致性: {(result - verified_result).abs().max().item():.8f}")

print(f"\n--- 模型完整性校验 ---")
integrity = checker.verify_model('classifier_v1', model)
print(f"  模型: {integrity['model_id']}")
print(f"  完整性: {'✓ 通过' if integrity['integrity_ok'] else '✗ 被篡改'}")
print(f"  哈希: {integrity['current_hash']}")

tampered_model = SimpleClassifier(dim=32, n_classes=5)
with torch.no_grad():
    tampered_model.net[0].weight += 0.01
tampered = checker.verify_model('classifier_v1', tampered_model)
print(f"\n  篡改模型检测: {'✗ 被篡改' if not tampered['integrity_ok'] else '✓ 通过'}")
print(f"  原始哈希: {tampered['expected_hash']}")
print(f"  当前哈希: {tampered['current_hash']}")

print(f"\n=== 产业实践要点 ===")
print(f"1. TLS 1.3: 端云通信必须加密, 禁用TLS 1.0/1.1")
print(f"2. MAC校验: 防止中间特征和推理结果被篡改")
print(f"3. 特征噪声: 中间特征加噪防止梯度反演攻击")
print(f"4. 模型签名: 每次推理前校验模型完整性")
print(f"5. 证书固定(Certificate Pinning): 防止中间人攻击")

---
### 端侧AI平台安全

#### 端侧AI平台的安全层次

| 层次 | 安全机制 | 威胁 |
|------|---------|------|
| **硬件层** | TEE/安全启动 | 物理攻击、固件篡改 |
| **OS层** | 沙箱/SELinux | 恶意App、权限提升 |
| **框架层** | 模型加密/签名校验 | 模型窃取/篡改 |
| **应用层** | 输入校验/输出过滤 | 对抗攻击、内容安全 |
| **通信层** | TLS/证书固定 | 中间人攻击 |

#### 主流端侧AI平台安全特性

| 平台 | 安全启动 | TEE | 模型加密 | 沙箱 |
|------|---------|-----|---------|------|
| **iOS/CoreML** | ✓ | Secure Enclave | ✓ | App Sandbox |
| **Android/NNAPI** | ✓ | TrustZone | ✓ | App Sandbox |
| **Qualcomm/QNN** | ✓ | TrustZone | ✓ | HAP Sandbox |
| **MTK/NeuronPilot** | ✓ | TrustZone | ✓ | ✓ |

In [ ]:
@dataclass
class SecurityCheckResult:
    check_name: str
    passed: bool
    severity: str
    details: str

class PlatformSecurityAuditor:
    """端侧AI平台安全审计器"""
    def __init__(self, platform: str = 'android'):
        self.platform = platform
        self.checks: List[SecurityCheckResult] = []

    def check_secure_boot(self, enabled: bool = True) -> SecurityCheckResult:
        result = SecurityCheckResult('secure_boot', enabled, 'critical',
                                     'Secure boot enabled' if enabled else 'Secure boot DISABLED')
        self.checks.append(result)
        return result

    def check_tee_available(self, available: bool = True) -> SecurityCheckResult:
        result = SecurityCheckResult('tee_available', available, 'high',
                                     'TEE available' if available else 'TEE not available')
        self.checks.append(result)
        return result

    def check_model_encryption(self, encrypted: bool = True) -> SecurityCheckResult:
        result = SecurityCheckResult('model_encryption', encrypted, 'high',
                                     'Model encrypted at rest' if encrypted else 'Model NOT encrypted')
        self.checks.append(result)
        return result

    def check_tls_version(self, version: str = '1.3') -> SecurityCheckResult:
        ok = version in ('1.2', '1.3')
        result = SecurityCheckResult('tls_version', ok, 'critical',
                                     f'TLS {version}' if ok else f'TLS {version} INSECURE')
        self.checks.append(result)
        return result

    def check_certificate_pinning(self, enabled: bool = True) -> SecurityCheckResult:
        result = SecurityCheckResult('cert_pinning', enabled, 'high',
                                     'Certificate pinning enabled' if enabled else 'Cert pinning DISABLED')
        self.checks.append(result)
        return result

    def check_input_validation(self, enabled: bool = True) -> SecurityCheckResult:
        result = SecurityCheckResult('input_validation', enabled, 'medium',
                                     'Input validation enabled' if enabled else 'Input validation DISABLED')
        self.checks.append(result)
        return result

    def check_output_filter(self, enabled: bool = True) -> SecurityCheckResult:
        result = SecurityCheckResult('output_filter', enabled, 'medium',
                                     'Output content filter enabled' if enabled else 'Output filter DISABLED')
        self.checks.append(result)
        return result

    def generate_report(self) -> Dict:
        total = len(self.checks)
        passed = sum(1 for c in self.checks if c.passed)
        critical_fail = any(not c.passed and c.severity == 'critical' for c in self.checks)
        return {
            'platform': self.platform, 'total_checks': total, 'passed': passed,
            'pass_rate': passed / total if total > 0 else 0,
            'critical_failure': critical_fail,
            'overall_status': 'FAIL' if critical_fail else ('PASS' if passed == total else 'WARNING'),
        }

auditor = PlatformSecurityAuditor('android')

print("=== 端侧AI平台安全审计 ===")
auditor.check_secure_boot(True)
auditor.check_tee_available(True)
auditor.check_model_encryption(True)
auditor.check_tls_version('1.3')
auditor.check_certificate_pinning(True)
auditor.check_input_validation(True)
auditor.check_output_filter(True)

print(f"\n{'检查项':<20} {'状态':<8} {'严重性':<10} {'详情'}")
print("-" * 60)
for check in auditor.checks:
    status = '✓' if check.passed else '✗'
    print(f"{check.check_name:<20} {status:<8} {check.severity:<10} {check.details}")

report = auditor.generate_report()
print(f"\n--- 审计报告 ---")
print(f"平台: {report['platform']}")
print(f"通过率: {report['pass_rate']:.0%} ({report['passed']}/{report['total_checks']})")
print(f"严重失败: {'是' if report['critical_failure'] else '否'}")
print(f"综合状态: {report['overall_status']}")

print(f"\n--- 不安全配置示例 ---")
bad_auditor = PlatformSecurityAuditor('android')
bad_auditor.check_secure_boot(False)
bad_auditor.check_tee_available(False)
bad_auditor.check_model_encryption(False)
bad_auditor.check_tls_version('1.0')
bad_auditor.check_certificate_pinning(False)
bad_auditor.check_input_validation(False)
bad_auditor.check_output_filter(False)
bad_report = bad_auditor.generate_report()
print(f"  通过率: {bad_report['pass_rate']:.0%}")
print(f"  综合状态: {bad_report['overall_status']}")
print(f"  严重失败: {'是' if bad_report['critical_failure'] else '否'}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 安全启动: 防止固件被篡改, 是所有安全的基础")
print(f"2. TEE: 敏感推理(人脸/支付)必须在TEE中执行")
print(f"3. 模型加密: AES-256加密模型文件, 运行时解密到TEE")
print(f"4. TLS 1.3: 端云通信必须使用TLS 1.3+证书固定")
print(f"5. 输入校验: 防止对抗攻击(输入范围/格式检查)")
print(f"6. 输出过滤: 防止模型输出敏感/有害内容")

---
### 合规框架

#### 主要AI隐私法规

| 法规 | 地区 | 核心要求 | 对端侧AI的影响 |
|------|------|---------|---------------|
| **GDPR** | 欧盟 | 数据最小化、用户同意、被遗忘权 | 数据本地处理、用户可删除数据 |
| **CCPA** | 加州 | 消费者知情权、选择权 | 披露数据使用、提供退出选项 |
| **个人信息保护法** | 中国 | 最小必要、知情同意 | 数据不出境、本地化处理 |
| **AI Act** | 欧盟 | 高风险AI系统监管 | 透明度、可解释性、人类监督 |
| **NIST AI RMF** | 美国 | AI风险管理框架 | 风险评估、监控、文档化 |

#### 端侧AI合规的关键要求

1. **数据最小化**：仅收集必要数据，端侧处理优先
2. **用户同意**：明确告知数据使用方式，获得用户授权
3. **数据本地化**：敏感数据不出设备，仅上传脱敏结果
4. **可审计性**：推理决策可追溯，满足可解释性要求
5. **被遗忘权**：用户可删除所有个人数据
6. **透明度**：披露AI使用方式、模型能力边界

In [ ]:
@dataclass
class ComplianceCheck:
    regulation: str
    requirement: str
    status: str
    evidence: str
    risk_level: str

class ComplianceFramework:
    """端侧AI合规框架: 自动化合规检查"""
    REGULATIONS = {
        'GDPR': [
            ('data_minimization', '数据最小化', '仅收集必要数据'),
            ('user_consent', '用户同意', '明确告知并获取授权'),
            ('right_to_erasure', '被遗忘权', '用户可删除所有个人数据'),
            ('data_localization', '数据本地化', '敏感数据不出设备'),
            ('transparency', '透明度', '披露AI使用方式'),
            ('dp_protection', '隐私保护', '差分隐私或等效保护'),
        ],
        'AI_Act': [
            ('risk_assessment', '风险评估', '评估AI系统风险等级'),
            ('human_oversight', '人类监督', '关键决策需人类确认'),
            ('explainability', '可解释性', '推理决策可追溯'),
            ('accuracy_monitoring', '准确性监控', '持续监控模型精度'),
            ('bias_detection', '偏见检测', '检测和缓解模型偏见'),
        ],
        'PIPL': [
            ('min_necessary', '最小必要', '仅处理必要个人信息'),
            ('informed_consent', '知情同意', '告知处理目的并获同意'),
            ('data_localization_cn', '数据不出境', '个人信息不出境'),
            ('security_measures', '安全措施', '采取必要安全保护措施'),
        ],
    }

    def __init__(self):
        self.checks: List[ComplianceCheck] = []
        self.compliance_status: Dict[str, Dict] = {}

    def audit(self, regulation: str, config: Dict) -> List[ComplianceCheck]:
        """对指定法规进行合规审计"""
        if regulation not in self.REGULATIONS:
            return []
        results = []
        for req_id, req_name, req_desc in self.REGULATIONS[regulation]:
            is_compliant = config.get(req_id, False)
            risk = 'high' if not is_compliant and req_id in ('user_consent', 'data_localization', 'min_necessary') else 'medium'
            results.append(ComplianceCheck(
                regulation=regulation, requirement=req_name,
                status='compliant' if is_compliant else 'non_compliant',
                evidence=req_desc if is_compliant else f'未满足: {req_desc}',
                risk_level=risk if not is_compliant else 'low'
            ))
        self.checks.extend(results)
        passed = sum(1 for c in results if c.status == 'compliant')
        self.compliance_status[regulation] = {
            'total': len(results), 'passed': passed,
            'pass_rate': passed / len(results),
            'high_risk_count': sum(1 for c in results if c.risk_level == 'high'),
        }
        return results

    def generate_compliance_report(self) -> Dict:
        """生成综合合规报告"""
        total = len(self.checks)
        passed = sum(1 for c in self.checks if c.status == 'compliant')
        high_risk = sum(1 for c in self.checks if c.risk_level == 'high')
        return {
            'total_checks': total, 'passed': passed, 'pass_rate': passed / total if total > 0 else 0,
            'high_risk_items': high_risk,
            'regulations': self.compliance_status,
            'overall_status': 'COMPLIANT' if high_risk == 0 and passed == total
                              else 'AT_RISK' if high_risk > 0 else 'PARTIAL',
        }

framework = ComplianceFramework()

gdpr_config = {
    'data_minimization': True, 'user_consent': True, 'right_to_erasure': True,
    'data_localization': True, 'transparency': True, 'dp_protection': False,
}
ai_act_config = {
    'risk_assessment': True, 'human_oversight': True, 'explainability': False,
    'accuracy_monitoring': True, 'bias_detection': False,
}
pipl_config = {
    'min_necessary': True, 'informed_consent': True,
    'data_localization_cn': True, 'security_measures': True,
}

print("=== 端侧AI合规审计 ===")
for reg_name, config in [('GDPR', gdpr_config), ('AI Act', ai_act_config), ('PIPL', pipl_config)]:
    results = framework.audit(reg_name, config)
    status = framework.compliance_status[reg_name]
    print(f"\n--- {reg_name} ({status['pass_rate']:.0%} 合规) ---")
    print(f"{'要求':<15} {'状态':<15} {'风险':<8} {'详情'}")
    print("-" * 60)
    for r in results:
        status_str = '✓ 合规' if r.status == 'compliant' else '✗ 不合规'
        print(f"{r.requirement:<15} {status_str:<15} {r.risk_level:<8} {r.evidence}")

report = framework.generate_compliance_report()
print(f"\n--- 综合合规报告 ---")
print(f"总检查项: {report['total_checks']}")
print(f"通过率: {report['pass_rate']:.0%}")
print(f"高风险项: {report['high_risk_items']}")
print(f"综合状态: {report['overall_status']}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 合规是上线前提: 不合规的产品不能发布")
print(f"2. 自动化审计: 每次模型更新后自动运行合规检查")
print(f"3. 数据分类: 根据敏感度分级处理(公开/内部/机密/绝密)")
print(f"4. 隐私影响评估(PIA): 新功能上线前必须完成PIA")
print(f"5. 合规文档: 保留合规审计记录, 应对监管审查")
print(f"6. 跨境合规: 中国数据不出境, 欧盟数据在欧盟处理")

---
## 总结与最佳实践

### 端侧AI安全的核心原则

1. **纵深防御**：硬件(TEE)→OS(沙箱)→框架(加密)→应用(校验)，多层防护
2. **数据最小化**：端侧处理优先，仅上传必要信息
3. **零信任**：不信任任何外部输入，所有通信加密+校验
4. **可审计**：所有安全决策可追溯，满足合规要求

### 安全部署Checklist

- [ ] 启用安全启动（Secure Boot）
- [ ] 敏感推理在TEE中执行
- [ ] 模型文件AES-256加密存储
- [ ] 端云通信使用TLS 1.3+证书固定
- [ ] 中间特征加噪防止梯度反演
- [ ] 模型水印保护IP
- [ ] 差分隐私训练（ε≤8）
- [ ] 输入校验防止对抗攻击
- [ ] 输出过滤防止有害内容
- [ ] 合规审计通过（GDPR/AI Act/PIPL）
- [ ] 隐私影响评估(PIA)完成
- [ ] 安全事件响应预案就绪